# Spine 2 · Knowledge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnakhoa/llm-engineering-playground/blob/main/spine/02-knowledge/knowledge.ipynb)

Your Flagship Agent learns to answer from Acme Notes' **Knowledge Base** and to
remember what a customer said earlier in the same Case. You will run three
Cases whose right answer is in a policy article, grade them with Checks, and
see why the Spine 1 loop cannot pass the multi-turn one.

Everything here runs **Offline**: replayed from reviewed recordings, no API key.
The lesson that goes with this notebook is `spine/02-knowledge/README.md`.

In [ ]:
# Setup. Locally this finds the course folder. On Colab it clones the course
# into the current folder once, and finds that clone again when you re-run this
# cell or restart the runtime.
import subprocess
import sys
from pathlib import Path

COURSE_URL = "https://github.com/dnakhoa/llm-engineering-playground.git"
CLONE = Path.cwd() / "llm-engineering-playground"


def find_course(start):
    for folder in [start, *start.parents, CLONE]:
        if (folder / "company").is_dir() and (folder / "llm").is_dir():
            return folder
    return None


COURSE = find_course(Path.cwd())
if COURSE is None:
    subprocess.run(["git", "clone", "--depth", "1", COURSE_URL, str(CLONE)], check=True)
    COURSE = CLONE
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("Course at", COURSE)

## 1. The Knowledge Base, and a retriever with no embeddings

The Knowledge Base is a folder of markdown articles. The default retriever is
lexical: BM25 over the articles' words, in pure Python. No API, no key, and the
same answer every time.

In [ ]:
from company.knowledge import LexicalRetriever, load_articles

articles = load_articles()
for article in articles:
    print(article.id, "-", article.title)
print()

retriever = LexicalRetriever(articles)
for question in ["Can I get a refund for this month?", "How many notebooks does Pro allow?"]:
    print(question)
    for hit in retriever.search(question, k=2):
        print("   {:.3f}  {}".format(hit.score, hit.article.id))

## 2. The new Action: `issue_refund`

The refund policy says: refunds within 30 days of the invoice, prorated by
unused days, rounded down to the cent, at most $200.00. The article is what
the agent reads. The Action is what enforces it, whatever the agent was told.

In [ ]:
from company.backend import Backend
from llm.types import ToolCall


def refund(account_id, invoice_id, amount_usd):
    actions = Backend.seeded().actions_for(account_id)
    return actions.run(ToolCall("c1", "issue_refund", {
        "account_id": account_id, "invoice_id": invoice_id, "amount_usd": amount_usd}))


print(refund("acct_1002", "inv_2002", 12.00).content)   # the whole month: too much
print(refund("acct_1003", "inv_3001", 100.00).content)  # an invoice from 45 days ago
print(refund("acct_1002", "inv_2002", 9.20).content)    # 12.00 x 23 unused / 30 days

## 3. Each Case declares its Actions

A Case offers the agent only the Actions it lists. The Spine 1 Case never
offered `issue_refund`, so its request, and the recording that replays it, are
exactly what they were before the Action existed.

In [ ]:
from company.runner import load_case

for case_id in ["upgrade-to-pro", "downgrade-with-prorated-refund",
                "annual-refund-outside-window", "upgrade-after-a-question"]:
    case = load_case(case_id)
    print("{:<32} actions={} knowledge_base={}".format(
        case.id, list(case.actions), case.knowledge_base))

## 4. A refund whose amount is in the policy

The customer asks for "whatever we're owed". Only the refund policy says how
much that is. The Checks hold the agent to the exact Backend change the Case
expects: the plan on Free, and one refund of $9.20.

In [ ]:
from checks.spine_2_knowledge import CHECKS
from company.runner import run_case


def show(outcome):
    for message in outcome.transcript:
        if message.text:
            print("[{}] {}".format(message.role, message.text))
        for call in message.tool_calls:
            print("[{}] calls {}({})".format(message.role, call.name, dict(call.arguments)))
        for result in message.tool_results:
            print("[tool] {}".format(result.content.splitlines()[0]))
    print()
    for check in CHECKS:
        result = check(outcome)
        print("PASS" if result.passed else "FAIL", "-", result.name, "-", result.detail)


refund = run_case(load_case("downgrade-with-prorated-refund"), "flagship.knowledge:run")
show(refund)

And a refund the policy does not allow at all: the right Action is none.

In [ ]:
window = run_case(load_case("annual-refund-outside-window"), "flagship.knowledge:run")
show(window)

## 5. Memory across a Case's turns

The Spine 2 agent is the Spine 1 loop with one change: the transcript lives in
`env.memory`, which the Case runner keeps for the whole Case.

In [ ]:
import inspect

import flagship.knowledge

print(inspect.getsource(flagship.knowledge.run))

remembers = run_case(load_case("upgrade-after-a-question"), "flagship.knowledge:run")
show(remembers)

The Spine 1 loop starts every turn from an empty transcript, so its second
request holds only "please go ahead", with no account in it. No recording of
this Case has that request, and the replay says so.

In [ ]:
from llm.replay import ReplayMismatchError

forgot = None
try:
    run_case(load_case("upgrade-after-a-question"), "flagship.loop:run")
except ReplayMismatchError as error:
    forgot = error
print(str(forgot).splitlines()[0])

## 6. The upgrade: embeddings behind the same interface

`EmbeddingsRetriever` ranks by cosine similarity between embedding vectors, so
"money back" can find "refund". It takes any `embed` function. The toy one
below counts a few words so it runs here for free; in practice you pass your
provider's embeddings endpoint. A live embeddings call costs money and changes
with the model, which is why the Offline Checks use the lexical retriever.

In [ ]:
import math

from company.knowledge import EmbeddingsRetriever

TOPICS = [("refund", "money back", "refunded"), ("plan", "pro", "team"),
          ("notebook",), ("invoice", "billing"), ("export",)]


def toy_embed(texts):
    vectors = []
    for text in texts:
        text = text.lower()
        vector = [float(sum(text.count(word) for word in topic)) for topic in TOPICS]
        norm = math.sqrt(sum(x * x for x in vector)) or 1.0
        vectors.append([x / norm for x in vector])
    return vectors


embedded = EmbeddingsRetriever(articles, embed=toy_embed)
for name, r in [("lexical", retriever), ("embeddings", embedded)]:
    print(name, [hit.article.id for hit in r.search("I want my money back", k=2)])

## 7. The Checks, cumulatively

Running module 2 runs module 1's Checks too.

In [ ]:
from checks.cli import run_checks

report = run_checks(through=2, agent="flagship.knowledge:run")

## Your turn

* Give your own agent memory, and run it live on the multi-turn Case:
  `run_case(load_case("upgrade-after-a-question"), my_run, mode="live")`.
* Change `SYSTEM` in your agent and replay any Case Offline. The mismatch names
  `body.system`: a new system prompt means re-recording every Case.
* Write a Case where the policy decides: a refund above $200.00, say. What
  should the expected Backend change be?